# Annotation Tensor Inspector

This notebook allows you to explore the annotation tensor described in the paper:
> **Beyond Black-Box Labels: Interpretable Criteria for Diagnosing Subjective NLP Tasks**

The tensor has shape **(N_sentences × N_criteria × N_models)**:
- `1`  = model answered **Oui** (criterion present)
- `0`  = model answered **Non** (criterion absent)
- `-1` = missing / could not be parsed

**Criteria mapping (see Section 4.1 of the paper):**
| Index | ID  | Name | Category |
|-------|-----|------|----------|
| 0 | q01 | Cost Reduction | c1: Performance & Efficiency |
| 1 | q02 | Operational Efficiency | c1: Performance & Efficiency |
| 2 | q03 | Organizational Impact | c1: Performance & Efficiency |
| 3 | q04 | User Well-being | c2: User Experience & Brand Value |
| 4 | q05 | Reputation & Recognition | c2: User Experience & Brand Value |
| 5 | q06 | Tangible/Perceived Quality | c2: User Experience & Brand Value |
| 6 | q07 | Regulatory Compliance | c3: Obligation & Safety |
| 7 | q08 | Risk Prevention/Security | c3: Obligation & Safety |
| 8 | q09 | Mandatory Requirement | c3: Obligation & Safety |

**Models (see Section 4.2 of the paper):**
| Index | Model |
|-------|-------|
| 0 | google/gemini-2.0-flash-001 |
| 1 | gpt-4.1-mini |
| 2 | gpt-4.1 |
| 3 | meta-llama/llama-3.3-70b-instruct |
| 4 | mistralai/mistral-large-2411 |
| 5 | qwen/qwen-2.5-72b-instruct |

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────
TENSOR_DIR  = Path("../data/tensors")
MAPPING_CSV = TENSOR_DIR / "mapping_sentences.csv"
TENSOR_NPY  = TENSOR_DIR / "tensor_raw_4700.npy"

CRITERIA = [
    "q01_CostReduction",
    "q02_OperationalEfficiency",
    "q03_OrganizationalImpact",
    "q04_UserWellbeing",
    "q05_ReputationRecognition",
    "q06_PerceivedQuality",
    "q07_RegulatoryCompliance",
    "q08_RiskPrevention",
    "q09_MandatoryRequirement",
]

MODELS = [
    "gemini-2.0-flash-001",
    "gpt-4.1-mini",
    "gpt-4.1",
    "llama-3.3-70b",
    "mistral-large-2411",
    "qwen-2.5-72b",
]

# ── Load ──────────────────────────────────────────────────────────
tensor  = np.load(TENSOR_NPY)
mapping = pd.read_csv(MAPPING_CSV)

print(f"Tensor shape : {tensor.shape}  (sentences × criteria × models)")
print(f"Sentences    : {len(mapping)}")
print(f"Criteria     : {len(CRITERIA)}")
print(f"Models       : {len(MODELS)}")

Tensor shape : (4700, 11, 6)  (sentences × criteria × models)
Sentences    : 4700
Criteria     : 9
Models       : 6


## 1. Inspect a single sentence

In [5]:
# Change this index to inspect any sentence (0 to 4698)
SENTENCE_IDX = 42

row = mapping.loc[mapping["tensor_idx"] == SENTENCE_IDX].iloc[0]
print(f"Sentence ID  : {row['sentence_id']}")
print(f"Client       : {row['client_uid']}")
print(f"Document     : {row['doc_uid']}")
print(f"Text         : {row['sentence']}")
print()

votes = tensor[SENTENCE_IDX, :9, :]  # shape: (9 criteria, 6 models)
df_votes = pd.DataFrame(votes, index=CRITERIA, columns=MODELS)
df_votes = df_votes.replace({1: "Oui", 0: "Non", -1: "Missing"})
print(df_votes)

Sentence ID  : 43
Client       : C002
Document     : D04
Text         : The consulting firm offers innovative solutions.

                          gemini-2.0-flash-001 gpt-4.1-mini gpt-4.1  \
q01_CostReduction                          Non          Non     Non   
q02_OperationalEfficiency                  Non          Non     Non   
q03_OrganizationalImpact                   Non          Non     Non   
q04_UserWellbeing                          Non          Non     Non   
q05_ReputationRecognition                  Oui          Oui     Oui   
q06_PerceivedQuality                       Oui          Non     Non   
q07_RegulatoryCompliance                   Non          Non     Non   
q08_RiskPrevention                         Non          Non     Non   
q09_MandatoryRequirement                   Non          Non     Non   

                          llama-3.3-70b mistral-large-2411 qwen-2.5-72b  
q01_CostReduction                   Non                Non          Non  
q02_OperationalEffi